<a href="https://colab.research.google.com/github/eshan14git/football-qa-nlp/blob/eshan-dev/notebooks/06_football_qa_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 - Clone the correct GitHub branch

!git clone -b eshan-dev https://github.com/eshan14git/football-qa-nlp.git

Cloning into 'football-qa-nlp'...
remote: Enumerating objects: 209, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 209 (delta 52), reused 46 (delta 21), pack-reused 112 (from 4)
Receiving objects: 100% (209/209), 38.21 MiB | 17.20 MiB/s, done.
Resolving deltas: 100% (84/84), done.


In [2]:
# Cell 2 - Import libraries

import os
import re
import joblib
import pandas as pd

project_path = "/content/football-qa-nlp"
data_path = os.path.join(project_path, "data")
rf_path = os.path.join(project_path, "models", "random_forest")

print("Setup complete.")

Setup complete.


In [3]:
# Cell 3 - Load Random Forest model and TF-IDF vectorizer

rf_model = joblib.load(
    os.path.join(rf_path, "trained_random_forest.pkl")
)

rf_vectorizer = joblib.load(
    os.path.join(rf_path, "random_forest_tfidf_vectorizer.pkl")
)

print("Random Forest model loaded.")
print("TF-IDF vectorizer loaded.")

Random Forest model loaded.
TF-IDF vectorizer loaded.


In [4]:
# Cell 4 - Load cleaned datasets

results_df = pd.read_csv(
    os.path.join(data_path, "results_clean.csv")
)

goalscorers_df = pd.read_csv(
    os.path.join(data_path, "goalscorers_clean.csv")
)

shootouts_df = pd.read_csv(
    os.path.join(data_path, "shootouts_clean.csv")
)

former_names_df = pd.read_csv(
    os.path.join(data_path, "former_names_clean.csv")
)

print("Datasets loaded successfully.")
print("Results:", results_df.shape)
print("Goalscorers:", goalscorers_df.shape)
print("Shootouts:", shootouts_df.shape)
print("Former names:", former_names_df.shape)

Datasets loaded successfully.
Results: (49485, 9)
Goalscorers: (47855, 8)
Shootouts: (682, 5)
Former names: (36, 4)


In [5]:
# Cell 5 - Basic intent prediction function

def predict_intent(question):
    question_vector = rf_vectorizer.transform([question])
    prediction = rf_model.predict(question_vector)[0]
    return prediction

test_question = "Who won the 2014 FIFA World Cup Final?"

print("Question:", test_question)
print("Predicted intent:", predict_intent(test_question))

Question: Who won the 2014 FIFA World Cup Final?
Predicted intent: match_winner


In [6]:
# Cell 6 - Prepare searchable team and tournament lists

all_teams = sorted(
    set(results_df["home_team"].dropna().tolist()) |
    set(results_df["away_team"].dropna().tolist())
)

all_tournaments = sorted(
    results_df["tournament"].dropna().unique().tolist()
)

print("Unique teams:", len(all_teams))
print("Unique tournaments:", len(all_tournaments))

Unique teams: 336
Unique tournaments: 200


In [7]:
# Cell 7 - Extract basic entities from a question

def extract_basic_entities(question):
    question_lower = question.lower()

    # Find teams mentioned in the question
    found_teams = [
        team for team in all_teams
        if team.lower() in question_lower
    ]

    # Find year
    year_match = re.search(r"\b(18|19|20)\d{2}\b", question)
    year = year_match.group() if year_match else None

    # Find tournament
    found_tournaments = [
        tournament for tournament in all_tournaments
        if tournament.lower() in question_lower
    ]

    return {
        "teams": found_teams,
        "year": year,
        "tournaments": found_tournaments
    }

In [8]:
# Cell 8 - Test entity extraction

test_questions = [
    "Who won between India and Myanmar?",
    "Who won between India and Myanmar in 2017?",
    "Who won the FIFA World Cup match between Germany and Argentina in 2014?",
    "What was the score between Brazil and Argentina in Copa América?"
]

for q in test_questions:
    print("\nQuestion:", q)
    print(extract_basic_entities(q))


Question: Who won between India and Myanmar?
{'teams': ['India', 'Myanmar'], 'year': None, 'tournaments': []}

Question: Who won between India and Myanmar in 2017?
{'teams': ['India', 'Myanmar'], 'year': '2017', 'tournaments': []}

Question: Who won the FIFA World Cup match between Germany and Argentina in 2014?
{'teams': ['Argentina', 'Germany'], 'year': '2014', 'tournaments': ['FIFA World Cup']}

Question: What was the score between Brazil and Argentina in Copa América?
{'teams': ['Argentina', 'Brazil'], 'year': None, 'tournaments': ['Copa América']}


In [9]:
# Cell 9 - Find candidate matches from extracted entities

def find_result_candidates(question):
    entities = extract_basic_entities(question)

    candidates = results_df.copy()

    teams = entities["teams"]
    year = entities["year"]
    tournaments = entities["tournaments"]

    # Filter by two detected teams
    if len(teams) >= 2:
        team1, team2 = teams[:2]

        candidates = candidates[
            (
                (candidates["home_team"] == team1) &
                (candidates["away_team"] == team2)
            )
            |
            (
                (candidates["home_team"] == team2) &
                (candidates["away_team"] == team1)
            )
        ]

    # Filter by year
    if year:
        candidates = candidates[
            candidates["date"].astype(str).str.startswith(year)
        ]

    # Filter by tournament
    if tournaments:
        tournament = tournaments[0]

        candidates = candidates[
            candidates["tournament"].str.lower() == tournament.lower()
        ]

    return candidates

In [10]:
# Cell 10 - Test candidate retrieval

question = "Who won between India and Myanmar in 2017?"

candidates = find_result_candidates(question)

print("Candidate matches found:", len(candidates))

display(
    candidates[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ]
)

Candidate matches found: 2


,date,home_team,away_team,home_score,away_score,tournament,city,country
40535,2017-03-28,Myanmar,India,0,1,AFC Asian Cup qualification,Yangon,Myanmar
41204,2017-11-14,India,Myanmar,2,2,AFC Asian Cup qualification,Margao,India


In [11]:
# Cell 11 - Inspect differences between candidate matches

def inspect_candidate_differences(candidates):
    if len(candidates) <= 1:
        return {}

    differences = {}

    fields = [
        "date",
        "tournament",
        "city",
        "country",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "neutral"
    ]

    for field in fields:
        unique_values = candidates[field].dropna().astype(str).unique().tolist()

        if len(unique_values) > 1:
            differences[field] = unique_values

    return differences

In [12]:
# Cell 12 - Test ambiguity inspection

question = "Who won between India and Myanmar in 2017?"

candidates = find_result_candidates(question)

differences = inspect_candidate_differences(candidates)

print("Candidate matches:", len(candidates))
print("\nDifferences:")

for field, values in differences.items():
    print(f"- {field}: {values}")

Candidate matches: 2

Differences:
- date: ['2017-03-28', '2017-11-14']
- city: ['Yangon', 'Margao']
- country: ['Myanmar', 'India']
- home_team: ['Myanmar', 'India']
- away_team: ['India', 'Myanmar']
- home_score: ['0', '2']
- away_score: ['1', '2']


In [13]:
# Cell 13 - Generate a clarification question

def generate_clarification_question(candidates):
    if len(candidates) == 0:
        return "I couldn't find a matching game."

    if len(candidates) == 1:
        return None

    differences = inspect_candidate_differences(candidates)

    # Prefer tournament when tournaments differ
    if "tournament" in differences:
        options = differences["tournament"]
        return (
            "I found multiple matching games. "
            "Do you remember the tournament? "
            + "Options: "
            + ", ".join(options)
        )

    # Then country/location
    if "country" in differences:
        options = differences["country"]
        return (
            "I found multiple matching games. "
            "Do you remember which country the game was played in? "
            + "Options: "
            + ", ".join(options)
        )

    # Then city
    if "city" in differences:
        options = differences["city"]
        return (
            "I found multiple matching games. "
            "Do you remember the city? "
            + "Options: "
            + ", ".join(options)
        )

    # Then home team
    if "home_team" in differences:
        options = differences["home_team"]
        return (
            "I found multiple matching games. "
            "Do you remember which team was the home team? "
            + "Options: "
            + ", ".join(options)
        )

    # Then exact date only as a last resort
    if "date" in differences:
        options = differences["date"]
        return (
            "I found multiple matching games. "
            "Do you remember which date it was? "
            + "Options: "
            + ", ".join(options)
        )

    return (
        "I found multiple matching games, but I need one more detail "
        "to identify the correct one."
    )

In [14]:
# Cell 14 - Test clarification generation

question = "Who won between India and Myanmar in 2017?"

candidates = find_result_candidates(question)

clarification = generate_clarification_question(candidates)

print("Question:", question)
print("\nAssistant:", clarification)

Question: Who won between India and Myanmar in 2017?

Assistant: I found multiple matching games. Do you remember which country the game was played in? Options: Myanmar, India


In [15]:
question = "Who won between India and Myanmar in 2017?"

candidates = find_result_candidates(question)

print("Candidate matches:", len(candidates))
print(generate_clarification_question(candidates))

Candidate matches: 2
I found multiple matching games. Do you remember which country the game was played in? Options: Myanmar, India


In [16]:
# Cell 15 - Apply a clarification reply to existing candidates

def apply_clarification(candidates, reply):
    reply_lower = reply.lower().strip()

    filtered = candidates.copy()

    # Year
    year_match = re.search(r"\b(18|19|20)\d{2}\b", reply)
    if year_match:
        year = year_match.group()
        filtered = filtered[
            filtered["date"].astype(str).str.startswith(year)
        ]

    # Tournament
    tournament_matches = [
        tournament
        for tournament in filtered["tournament"].dropna().unique()
        if tournament.lower() in reply_lower
    ]

    if tournament_matches:
        tournament = tournament_matches[0]
        filtered = filtered[
            filtered["tournament"].str.lower() == tournament.lower()
        ]

    # Country
    country_matches = [
        country
        for country in filtered["country"].dropna().unique()
        if str(country).lower() in reply_lower
    ]

    if country_matches:
        country = country_matches[0]
        filtered = filtered[
            filtered["country"].astype(str).str.lower() == str(country).lower()
        ]

    # City
    city_matches = [
        city
        for city in filtered["city"].dropna().unique()
        if str(city).lower() in reply_lower
    ]

    if city_matches:
        city = city_matches[0]
        filtered = filtered[
            filtered["city"].astype(str).str.lower() == str(city).lower()
        ]

    # Home team
    home_team_matches = [
        team
        for team in filtered["home_team"].dropna().unique()
        if str(team).lower() in reply_lower
    ]

    if home_team_matches:
        team = home_team_matches[0]
        filtered = filtered[
            filtered["home_team"].astype(str).str.lower() == str(team).lower()
        ]

    return filtered

In [17]:
# Cell 16 - Test clarification filtering

question = "Who won between India and Myanmar in 2017?"

candidates = find_result_candidates(question)

print("Initial candidates:", len(candidates))

clarification_reply = "India"

filtered_candidates = apply_clarification(
    candidates,
    clarification_reply
)

print("Candidates after clarification:", len(filtered_candidates))

display(
    filtered_candidates[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ]
)

Initial candidates: 2
Candidates after clarification: 1


,date,home_team,away_team,home_score,away_score,tournament,city,country
41204,2017-11-14,India,Myanmar,2,2,AFC Asian Cup qualification,Margao,India


In [18]:
# Cell 17 - Generate an answer from one selected result row

def generate_result_answer(intent, row):
    home_team = row["home_team"]
    away_team = row["away_team"]
    home_score = row["home_score"]
    away_score = row["away_score"]
    date = row["date"]
    tournament = row["tournament"]
    city = row["city"]
    country = row["country"]
    neutral = row["neutral"]

    if intent == "home_team_score":
        return f"{home_team} scored {home_score} goal(s)."

    elif intent == "away_team_score":
        return f"{away_team} scored {away_score} goal(s)."

    elif intent == "match_score":
        return f"{home_team} {home_score} - {away_score} {away_team}."

    elif intent == "match_winner":
        if home_score > away_score:
            return f"{home_team} won the match {home_score}-{away_score}."
        elif away_score > home_score:
            return f"{away_team} won the match {away_score}-{home_score}."
        else:
            return f"The match ended in a {home_score}-{away_score} draw."

    elif intent == "total_goals":
        total = home_score + away_score
        return f"A total of {total} goal(s) were scored."

    elif intent == "match_date":
        return f"The match was played on {date}."

    elif intent == "match_location":
        return f"The match was played in {city}, {country}."

    elif intent == "tournament":
        return f"The match was part of the {tournament}."

    elif intent == "neutral_status":
        neutral_text = "Yes" if bool(neutral) else "No"
        return f"{neutral_text}, the match {'was' if bool(neutral) else 'was not'} played at a neutral venue."

    return "Answer generation for this intent is not implemented yet."

In [19]:
# Cell 18 - Test result answer generation

question = "Who won between India and Myanmar in 2017?"

intent = predict_intent(question)

candidates = find_result_candidates(question)

print("Intent:", intent)
print("Initial candidates:", len(candidates))

# User clarification
reply = "India"

filtered_candidates = apply_clarification(
    candidates,
    reply
)

print("Candidates after clarification:", len(filtered_candidates))

if len(filtered_candidates) == 1:
    selected_match = filtered_candidates.iloc[0]
    answer = generate_result_answer(intent, selected_match)
    print("Answer:", answer)
else:
    print(generate_clarification_question(filtered_candidates))

Intent: match_winner
Initial candidates: 2
Candidates after clarification: 1
Answer: The match ended in a 2-2 draw.


In [27]:
# Cell 19 - Interactive Results QA conversation

RESULT_INTENTS = {
    "home_team_score",
    "away_team_score",
    "match_date",
    "match_location",
    "match_score",
    "match_winner",
    "neutral_status",
    "total_goals",
    "tournament"
}


def ask_results_question(question):
    # Step 1: Predict intent
    intent = predict_intent(question)

    print(f"\nPredicted intent: {intent}")

    # Make sure this function is only handling results-based intents
    if intent not in RESULT_INTENTS:
        print(
            "This question belongs to another data source "
            "and will be handled in a later stage."
        )
        return

    # Step 2: Find initial candidate matches
    candidates = find_result_candidates(question)

    print(f"Matching records found: {len(candidates)}")

    # Nothing found
    if len(candidates) == 0:
        print(
            "\nAssistant: I couldn't find a match that fits "
            "the information in your question."
        )
        return

    # Step 3: Resolve ambiguity conversationally
    while len(candidates) > 1:

        clarification = generate_smart_clarification(candidates)

        print(f"\nAssistant: {clarification}")

        reply = input("\nYou: ").strip()

        if not reply:
            print("\nAssistant: Please provide a little more information.")
            continue

        new_candidates = apply_clarification(candidates, reply)

        # Clarification didn't narrow anything
        if len(new_candidates) == len(candidates):
            print(
                "\nAssistant: That didn't narrow the matches down. "
                "Let's try another detail."
            )
            continue

        # Clarification accidentally removed everything
        if len(new_candidates) == 0:
            print(
                "\nAssistant: I couldn't match that detail to the "
                "remaining games. Please try another detail."
            )
            continue

        candidates = new_candidates

        print(f"\nRemaining matches: {len(candidates)}")

    # Step 4: Exactly one match remains
    selected_match = candidates.iloc[0]

    answer = generate_result_answer(
        intent,
        selected_match
    )

    print(f"\nAssistant: {answer}")

In [23]:
# Cell 20 - First interactive chatbot test

question = input("Ask a football question: ")

ask_results_question(question)

Ask a football question: who won against france vs argentina

Predicted intent: match_winner
Matching records found: 13

Assistant: I found matches across 12 different years. Do you remember roughly which year it was?

You: 2022

Remaining matches: 1

Assistant: The match ended in a 3-3 draw.


In [22]:
# Cell 21 - Smarter clarification generator

def generate_smart_clarification(candidates):
    if len(candidates) == 0:
        return "I couldn't find a matching game."

    if len(candidates) == 1:
        return None

    # Build useful derived fields
    temp = candidates.copy()
    temp["year"] = temp["date"].astype(str).str[:4]

    # 1. Prefer year if multiple years remain
    years = sorted(temp["year"].dropna().unique().tolist())

    if len(years) > 1:
        # Avoid dumping too many options
        if len(years) <= 8:
            return (
                "I found matches from multiple years. "
                "Do you remember the year? "
                f"Options: {', '.join(years)}"
            )

        return (
            f"I found matches across {len(years)} different years. "
            "Do you remember roughly which year it was?"
        )

    # 2. Tournament
    tournaments = sorted(
        temp["tournament"].dropna().astype(str).unique().tolist()
    )

    if len(tournaments) > 1:
        if len(tournaments) <= 8:
            return (
                "I found more than one possible match. "
                "Do you remember the tournament? "
                f"Options: {', '.join(tournaments)}"
            )

        return (
            "I found matches from several tournaments. "
            "Do you remember which competition it was?"
        )

    # 3. Country
    countries = sorted(
        temp["country"].dropna().astype(str).unique().tolist()
    )

    if len(countries) > 1:
        return (
            "Do you remember which country the match was played in? "
            f"Options: {', '.join(countries)}"
        )

    # 4. City
    cities = sorted(
        temp["city"].dropna().astype(str).unique().tolist()
    )

    if len(cities) > 1:
        return (
            "Do you remember the city where the match was played? "
            f"Options: {', '.join(cities)}"
        )

    # 5. Home team
    home_teams = sorted(
        temp["home_team"].dropna().astype(str).unique().tolist()
    )

    if len(home_teams) > 1:
        return (
            "Do you remember which team was listed as the home team? "
            f"Options: {', '.join(home_teams)}"
        )

    # 6. Exact date only as a last resort
    dates = sorted(
        temp["date"].dropna().astype(str).unique().tolist()
    )

    if len(dates) > 1:
        return (
            "I still found more than one possible match. "
            "Do you remember the exact date? "
            f"Options: {', '.join(dates)}"
        )

    return (
        "I still found multiple matching records. "
        "Can you give me one more detail about the match?"
    )

In [29]:

question = "Who won between Brazil and Argentina?"
ask_results_question(question)


Predicted intent: match_winner
Matching records found: 110

Assistant: I found matches across 64 different years. Do you remember roughly which year it was?

You: 2025

Remaining matches: 1

Assistant: Argentina won the match 4-1.


In [30]:
print("Model loaded:", "rf_model" in globals())
print("Results loaded:", "results_df" in globals())
print("Chatbot loaded:", "ask_results_question" in globals())
print("Smart clarification loaded:", "generate_smart_clarification" in globals())

Model loaded: True
Results loaded: True
Chatbot loaded: True
Smart clarification loaded: True


In [26]:
# Improved natural clarification filtering with accent normalization

def apply_smart_clarification(candidates, reply):
    reply_normalized = normalize_text(reply)
    filtered = candidates.copy()

    # -------------------------------------------------
    # 1. YEAR
    # -------------------------------------------------
    year_match = re.search(r"\b(18|19|20)\d{2}\b", reply)

    if year_match:
        year = year_match.group()

        year_filtered = filtered[
            filtered["date"].astype(str).str.startswith(year)
        ]

        if not year_filtered.empty:
            return year_filtered

    # -------------------------------------------------
    # 2. EXACT / NATURAL TOURNAMENT MATCH
    # -------------------------------------------------
    tournaments = (
        filtered["tournament"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    for tournament in tournaments:

        tournament_normalized = normalize_text(tournament)

        if tournament_normalized in reply_normalized:

            tournament_filtered = filtered[
                filtered["tournament"]
                .apply(normalize_text)
                == tournament_normalized
            ]

            if not tournament_filtered.empty:
                return tournament_filtered

    # -------------------------------------------------
    # 3. PARTIAL TOURNAMENT MATCH
    # -------------------------------------------------
    meaningful_words = [
        word
        for word in re.findall(r"[a-zA-Z]+", reply_normalized)
        if len(word) >= 4
    ]

    if meaningful_words:

        tournament_mask = filtered["tournament"].fillna("").apply(
            lambda value: all(
                word in normalize_text(value)
                for word in meaningful_words
            )
        )

        tournament_filtered = filtered[tournament_mask]

        if not tournament_filtered.empty:
            return tournament_filtered

    # -------------------------------------------------
    # 4. COUNTRY
    # -------------------------------------------------
    countries = (
        filtered["country"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    for country in countries:

        if normalize_text(country) in reply_normalized:

            country_filtered = filtered[
                filtered["country"].apply(normalize_text)
                == normalize_text(country)
            ]

            if not country_filtered.empty:
                return country_filtered

    # -------------------------------------------------
    # 5. CITY
    # -------------------------------------------------
    cities = (
        filtered["city"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    for city in cities:

        if normalize_text(city) in reply_normalized:

            city_filtered = filtered[
                filtered["city"].apply(normalize_text)
                == normalize_text(city)
            ]

            if not city_filtered.empty:
                return city_filtered

    # -------------------------------------------------
    # 6. HOME TEAM
    # -------------------------------------------------
    if "home" in reply_normalized:

        home_teams = (
            filtered["home_team"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        for team in home_teams:

            if normalize_text(team) in reply_normalized:

                home_filtered = filtered[
                    filtered["home_team"].apply(normalize_text)
                    == normalize_text(team)
                ]

                if not home_filtered.empty:
                    return home_filtered

    # Nothing understood
    return filtered

In [25]:
import unicodedata

def normalize_text(text):
    text = str(text).lower().strip()

    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )

    return text

print(normalize_text("Copa América"))
print(normalize_text("Copa America"))

copa america
copa america


In [31]:
question = "Who won between Brazil and Argentina?"

candidates = find_result_candidates(question)

print("Initial candidates:", len(candidates))

test_reply = "around 2021"

filtered = apply_smart_clarification(
    candidates,
    test_reply
)

print("Candidates after reply:", len(filtered))

display(
    filtered[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ]
)

Initial candidates: 110
Candidates after reply: 2


,date,home_team,away_team,home_score,away_score,tournament,city,country
44234,2021-07-10,Brazil,Argentina,0,1,Copa América,Rio de Janeiro,Brazil
44768,2021-11-16,Argentina,Brazil,0,0,FIFA World Cup qualification,San Juan,Argentina


In [32]:
reply = "the Copa America one"

filtered_again = apply_smart_clarification(
    filtered,
    reply
)

print("Candidates after tournament clarification:", len(filtered_again))

display(
    filtered_again[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ]
)

Candidates after tournament clarification: 1


,date,home_team,away_team,home_score,away_score,tournament,city,country
44234,2021-07-10,Brazil,Argentina,0,1,Copa América,Rio de Janeiro,Brazil


In [24]:
# Cell 24 - Text normalization helper

import unicodedata

def normalize_text(text):
    text = str(text).lower().strip()

    # Remove accents:
    # América -> America
    # Côte -> Cote
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )

    return text

In [28]:
# Cell 26 - Interactive Results chatbot with smart clarification

RESULT_INTENTS = {
    "home_team_score",
    "away_team_score",
    "match_date",
    "match_location",
    "match_score",
    "match_winner",
    "neutral_status",
    "total_goals",
    "tournament"
}


def ask_results_question(question):
    # Predict intent
    intent = predict_intent(question)

    print(f"\nPredicted intent: {intent}")

    if intent not in RESULT_INTENTS:
        print(
            "\nAssistant: This question belongs to another "
            "data source and will be handled later."
        )
        return

    # Find initial candidates
    candidates = find_result_candidates(question)

    print(f"Matching records found: {len(candidates)}")

    if len(candidates) == 0:
        print(
            "\nAssistant: I couldn't find a match that fits "
            "the information in your question."
        )
        return

    # Resolve ambiguity
    while len(candidates) > 1:

        clarification = generate_smart_clarification(candidates)

        print(f"\nAssistant: {clarification}")

        reply = input("\nYou: ").strip()

        if not reply:
            print(
                "\nAssistant: Please provide a little more information."
            )
            continue

        new_candidates = apply_smart_clarification(
            candidates,
            reply
        )

        # Reply did not narrow candidates
        if len(new_candidates) == len(candidates):
            print(
                "\nAssistant: That detail didn't narrow the matches down. "
                "Please try another detail."
            )
            continue

        # Reply removed every candidate
        if len(new_candidates) == 0:
            print(
                "\nAssistant: I couldn't match that detail to the "
                "remaining games. Please try another detail."
            )
            continue

        candidates = new_candidates

        print(f"\nRemaining matches: {len(candidates)}")

    # One exact match remains
    selected_match = candidates.iloc[0]

    answer = generate_result_answer(
        intent,
        selected_match
    )

    print(f"\nAssistant: {answer}")

In [33]:
# Cell 27 - Full smart conversation test

question = "Who won between Brazil and Argentina?"

ask_results_question(question)


Predicted intent: match_winner
Matching records found: 110

Assistant: I found matches across 64 different years. Do you remember roughly which year it was?

You: 2021

Remaining matches: 2

Assistant: I found more than one possible match. Do you remember the tournament? Options: Copa América, FIFA World Cup qualification

You: Copa america

Remaining matches: 1

Assistant: Argentina won the match 1-0.


In [34]:
GOALSCORER_INTENTS = {
    "scorer",
    "goal_minute",
    "own_goal_status",
    "penalty_status"
}

In [35]:
all_scorers = sorted(
    goalscorers_df["scorer"]
    .dropna()
    .astype(str)
    .unique()
    .tolist()
)

def add_match_key(df):
    temp = df.copy()

    temp["match_key"] = (
        temp["date"].astype(str)
        + "||"
        + temp["home_team"].astype(str)
        + "||"
        + temp["away_team"].astype(str)
    )

    return temp

goalscorers_with_key = add_match_key(goalscorers_df)
results_with_key = add_match_key(results_df)

print("Unique scorers:", len(all_scorers))

Unique scorers: 15360


In [36]:
def extract_goalscorer_entities(question):
    question_normalized = normalize_text(question)

    found_teams = [
        team
        for team in all_teams
        if normalize_text(team) in question_normalized
    ]

    found_players = [
        player
        for player in all_scorers
        if normalize_text(player) in question_normalized
    ]

    year_match = re.search(r"\b(18|19|20)\d{2}\b", question)
    year = year_match.group() if year_match else None

    return {
        "teams": found_teams,
        "players": found_players,
        "year": year
    }

In [42]:
# Cell 36 - Improved goalscorer candidate retrieval

def find_goalscorer_candidates(question):
    entities = extract_goalscorer_entities(question)

    candidates = goalscorers_with_key.copy()

    teams = entities["teams"]
    players = entities["players"]
    year = entities["year"]

    # -------------------------------------------------
    # 1. PLAYER FILTER
    # -------------------------------------------------
    if players:
        player = players[0]

        candidates = candidates[
            candidates["scorer"] == player
        ]

    # -------------------------------------------------
    # 2. TEAM / OPPONENT FILTERING
    # -------------------------------------------------

    # Two teams explicitly mentioned
    if len(teams) >= 2:
        team1, team2 = teams[:2]

        candidates = candidates[
            (
                (candidates["home_team"] == team1)
                & (candidates["away_team"] == team2)
            )
            |
            (
                (candidates["home_team"] == team2)
                & (candidates["away_team"] == team1)
            )
        ]

    # Only one team mentioned + player mentioned
    # Treat the team as the opponent/match participant
    elif len(teams) == 1 and players:
        mentioned_team = teams[0]

        candidates = candidates[
            (candidates["home_team"] == mentioned_team)
            |
            (candidates["away_team"] == mentioned_team)
        ]

    # -------------------------------------------------
    # 3. YEAR
    # -------------------------------------------------
    if year:
        candidates = candidates[
            candidates["date"].astype(str).str.startswith(year)
        ]

    # -------------------------------------------------
    # 4. SCORING TEAM
    # For wording like "scored for Paraguay"
    # -------------------------------------------------
    scoring_team = extract_scoring_team(question)

    if scoring_team:
        candidates = candidates[
            candidates["team"] == scoring_team
        ]

    return candidates

In [43]:
question = "Which player scored for Paraguay against Chile?"

intent = predict_intent(question)
candidates = find_goalscorer_candidates(question)

print("Predicted intent:", intent)
print("Goal records found:", len(candidates))
print("Scoring teams found:", candidates["team"].unique())

display(
    candidates[
        [
            "date",
            "home_team",
            "away_team",
            "team",
            "scorer",
            "minute",
            "own_goal",
            "penalty"
        ]
    ].head(20)
)

Predicted intent: scorer
Goal records found: 68
Scoring teams found: ['Paraguay']


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
106,1922-10-05,Chile,Paraguay,Paraguay,Julio Ramírez,5,False,False
107,1922-10-05,Chile,Paraguay,Paraguay,Ildefonso López,78,False,False
108,1922-10-05,Chile,Paraguay,Paraguay,Luis Fretes,86,False,False
244,1924-11-01,Chile,Paraguay,Paraguay,Ildefonso López,15,False,False
245,1924-11-01,Chile,Paraguay,Paraguay,Ildefonso López,33,False,False
246,1924-11-01,Chile,Paraguay,Paraguay,Gerardo Rivas,52,False,False
323,1926-11-03,Chile,Paraguay,Paraguay,Luis Vargas Peña,40,False,False
941,1937-01-17,Chile,Paraguay,Paraguay,Juan Amarilla,5,False,False
944,1937-01-17,Chile,Paraguay,Paraguay,Martín Flor,47,False,False
945,1937-01-17,Chile,Paraguay,Paraguay,Raúl Núñez Velloso,78,False,False


In [44]:
# Cell 33 - Inspect unique matches represented by goal records

unique_matches = (
    candidates[
        [
            "match_key",
            "date",
            "home_team",
            "away_team"
        ]
    ]
    .drop_duplicates()
    .sort_values("date")
)

print("Goal-event records:", len(candidates))
print("Unique matches:", len(unique_matches))

display(unique_matches.head(20))

Goal-event records: 68
Unique matches: 32


,match_key,date,home_team,away_team
106,1922-10-05||Chile||Paraguay,1922-10-05,Chile,Paraguay
244,1924-11-01||Chile||Paraguay,1924-11-01,Chile,Paraguay
323,1926-11-03||Chile||Paraguay,1926-11-03,Chile,Paraguay
941,1937-01-17||Chile||Paraguay,1937-01-17,Chile,Paraguay
1161,1939-01-15||Chile||Paraguay,1939-01-15,Chile,Paraguay
1287,1942-01-22||Chile||Paraguay,1942-01-22,Chile,Paraguay
1423,1946-01-19||Chile||Paraguay,1946-01-19,Chile,Paraguay
1556,1947-12-23||Chile||Paraguay,1947-12-23,Chile,Paraguay
1672,1949-04-27||Chile||Paraguay,1949-04-27,Chile,Paraguay
1935,1953-02-25||Chile||Paraguay,1953-02-25,Chile,Paraguay


In [45]:
# Cell 34 - Enrich candidate matches using results dataset

enriched_matches = unique_matches.merge(
    results_with_key[
        [
            "match_key",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country",
            "neutral"
        ]
    ],
    on="match_key",
    how="left"
)

print("Enriched matches:", len(enriched_matches))

display(
    enriched_matches[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ].head(20)
)

Enriched matches: 32


,date,home_team,away_team,home_score,away_score,tournament,city,country
0,1922-10-05,Chile,Paraguay,0,3,Copa América,Rio de Janeiro,Brazil
1,1924-11-01,Chile,Paraguay,1,3,Copa América,Montevideo,Uruguay
2,1926-11-03,Chile,Paraguay,5,1,Copa América,Santiago,Chile
3,1937-01-17,Chile,Paraguay,2,3,Copa América,Buenos Aires,Argentina
4,1939-01-15,Chile,Paraguay,1,5,Copa América,Lima,Peru
5,1942-01-22,Chile,Paraguay,0,2,Copa América,Montevideo,Uruguay
6,1946-01-19,Chile,Paraguay,2,1,Copa América,Buenos Aires,Argentina
7,1947-12-23,Chile,Paraguay,0,1,Copa América,Guayaquil,Ecuador
8,1949-04-27,Chile,Paraguay,2,4,Copa América,São Paulo,Brazil
9,1953-02-25,Chile,Paraguay,0,3,Copa América,Lima,Peru


In [41]:
# Cell 35 - Detect the scoring team from phrases like "for Paraguay"

def extract_scoring_team(question):
    question_normalized = normalize_text(question)

    # Look for patterns like:
    # "scored for Paraguay"
    # "goalscorer for Argentina"
    # "goal for Brazil"
    for team in all_teams:
        team_normalized = normalize_text(team)

        patterns = [
            f"for {team_normalized}",
            f"by {team_normalized}"
        ]

        if any(pattern in question_normalized for pattern in patterns):
            return team

    return None

In [46]:
test_questions = [
    "Which player scored for Paraguay against Chile?",
    "Who scored for Argentina against France?",
    "Who scored in Paraguay vs Chile?"
]

for q in test_questions:
    print(q)
    print("Scoring team:", extract_scoring_team(q))
    print()

Which player scored for Paraguay against Chile?
Scoring team: Paraguay

Who scored for Argentina against France?
Scoring team: Argentina

Who scored in Paraguay vs Chile?
Scoring team: None



In [47]:
# Cell 38 - Group goal events by unique match

def get_unique_goal_matches(candidates):
    return (
        candidates[
            [
                "match_key",
                "date",
                "home_team",
                "away_team"
            ]
        ]
        .drop_duplicates()
        .sort_values("date")
    )


question = "Which player scored for Paraguay against Chile?"

candidates = find_goalscorer_candidates(question)
unique_goal_matches = get_unique_goal_matches(candidates)

print("Goal records:", len(candidates))
print("Unique matches:", len(unique_goal_matches))

display(unique_goal_matches.head(20))

Goal records: 68
Unique matches: 32


,match_key,date,home_team,away_team
106,1922-10-05||Chile||Paraguay,1922-10-05,Chile,Paraguay
244,1924-11-01||Chile||Paraguay,1924-11-01,Chile,Paraguay
323,1926-11-03||Chile||Paraguay,1926-11-03,Chile,Paraguay
941,1937-01-17||Chile||Paraguay,1937-01-17,Chile,Paraguay
1161,1939-01-15||Chile||Paraguay,1939-01-15,Chile,Paraguay
1287,1942-01-22||Chile||Paraguay,1942-01-22,Chile,Paraguay
1423,1946-01-19||Chile||Paraguay,1946-01-19,Chile,Paraguay
1556,1947-12-23||Chile||Paraguay,1947-12-23,Chile,Paraguay
1672,1949-04-27||Chile||Paraguay,1949-04-27,Chile,Paraguay
1935,1953-02-25||Chile||Paraguay,1953-02-25,Chile,Paraguay


In [48]:
# Cell 39 - Enrich scorer match candidates with results data

def enrich_goal_matches(unique_matches):
    return unique_matches.merge(
        results_with_key[
            [
                "match_key",
                "home_score",
                "away_score",
                "tournament",
                "city",
                "country",
                "neutral"
            ]
        ],
        on="match_key",
        how="left"
    )


enriched_goal_matches = enrich_goal_matches(unique_goal_matches)

print("Enriched unique matches:", len(enriched_goal_matches))

display(
    enriched_goal_matches[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ].head(20)
)

Enriched unique matches: 32


,date,home_team,away_team,home_score,away_score,tournament,city,country
0,1922-10-05,Chile,Paraguay,0,3,Copa América,Rio de Janeiro,Brazil
1,1924-11-01,Chile,Paraguay,1,3,Copa América,Montevideo,Uruguay
2,1926-11-03,Chile,Paraguay,5,1,Copa América,Santiago,Chile
3,1937-01-17,Chile,Paraguay,2,3,Copa América,Buenos Aires,Argentina
4,1939-01-15,Chile,Paraguay,1,5,Copa América,Lima,Peru
5,1942-01-22,Chile,Paraguay,0,2,Copa América,Montevideo,Uruguay
6,1946-01-19,Chile,Paraguay,2,1,Copa América,Buenos Aires,Argentina
7,1947-12-23,Chile,Paraguay,0,1,Copa América,Guayaquil,Ecuador
8,1949-04-27,Chile,Paraguay,2,4,Copa América,São Paulo,Brazil
9,1953-02-25,Chile,Paraguay,0,3,Copa América,Lima,Peru


In [49]:
# Cell 40 - Clarification for scorer match candidates

def generate_goal_match_clarification(enriched_matches):
    if len(enriched_matches) == 0:
        return "I couldn't find a matching game."

    if len(enriched_matches) == 1:
        return None

    return generate_smart_clarification(enriched_matches)

In [50]:
# Cell 41 - Apply clarification to enriched scorer matches

def apply_goal_match_clarification(enriched_matches, reply):
    return apply_smart_clarification(
        enriched_matches,
        reply
    )

In [51]:
question = "Which player scored for Paraguay against Chile?"

goal_candidates = find_goalscorer_candidates(question)

unique_goal_matches = get_unique_goal_matches(
    goal_candidates
)

enriched_goal_matches = enrich_goal_matches(
    unique_goal_matches
)

print("Possible matches:", len(enriched_goal_matches))

clarification = generate_goal_match_clarification(
    enriched_goal_matches
)

print("\nAssistant:", clarification)

Possible matches: 32

Assistant: I found matches across 29 different years. Do you remember roughly which year it was?


In [52]:
reply = "around 2021"

filtered_goal_matches = apply_goal_match_clarification(
    enriched_goal_matches,
    reply
)

print("Remaining matches:", len(filtered_goal_matches))

display(
    filtered_goal_matches[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country"
        ]
    ]
)

Remaining matches: 1


,date,home_team,away_team,home_score,away_score,tournament,city,country
30,2021-06-24,Chile,Paraguay,0,2,Copa América,Brasília,Brazil


In [53]:
# Cell 44 - Retrieve goal events for the selected match

selected_match = filtered_goal_matches.iloc[0]
selected_match_key = selected_match["match_key"]

selected_goal_events = goal_candidates[
    goal_candidates["match_key"] == selected_match_key
].copy()

print("Goal records for selected match:", len(selected_goal_events))

display(
    selected_goal_events[
        [
            "date",
            "home_team",
            "away_team",
            "team",
            "scorer",
            "minute",
            "own_goal",
            "penalty"
        ]
    ]
)

Goal records for selected match: 2


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
40924,2021-06-24,Chile,Paraguay,Paraguay,Braian Samudio,33,False,False
40925,2021-06-24,Chile,Paraguay,Paraguay,Miguel Almirón,58,False,True


In [54]:
# Cell 45 - Generate answers for goalscorer-based intents

def generate_goalscorer_answer(intent, goal_events):
    if goal_events.empty:
        return "I couldn't find a matching goal record."

    home_team = goal_events.iloc[0]["home_team"]
    away_team = goal_events.iloc[0]["away_team"]
    date = goal_events.iloc[0]["date"]

    # -------------------------------------------------
    # SCORER
    # -------------------------------------------------
    if intent == "scorer":
        scorers = (
            goal_events["scorer"]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

        if len(scorers) == 1:
            return (
                f"{scorers[0]} scored in the "
                f"{home_team} vs {away_team} match on {date}."
            )

        return (
            f"The scorers were {', '.join(scorers)} "
            f"in the {home_team} vs {away_team} match on {date}."
        )

    # -------------------------------------------------
    # GOAL MINUTE
    # -------------------------------------------------
    elif intent == "goal_minute":
        details = []

        for _, row in goal_events.iterrows():
            details.append(
                f"{row['scorer']} in the {row['minute']}th minute"
            )

        return "The matching goal(s) were scored by " + "; ".join(details) + "."

    # -------------------------------------------------
    # OWN GOAL STATUS
    # -------------------------------------------------
    elif intent == "own_goal_status":
        details = []

        for _, row in goal_events.iterrows():
            status = "was" if bool(row["own_goal"]) else "was not"

            details.append(
                f"{row['scorer']}'s goal {status} an own goal"
            )

        return "; ".join(details) + "."

    # -------------------------------------------------
    # PENALTY STATUS
    # -------------------------------------------------
    elif intent == "penalty_status":
        details = []

        for _, row in goal_events.iterrows():
            status = "was" if bool(row["penalty"]) else "was not"

            details.append(
                f"{row['scorer']}'s goal {status} a penalty"
            )

        return "; ".join(details) + "."

    return "Answer generation for this intent is not available yet."

In [55]:
# Cell 46 - Test scorer answer generation

question = "Which player scored for Paraguay against Chile?"

intent = predict_intent(question)

answer = generate_goalscorer_answer(
    intent,
    selected_goal_events
)

print("Question:", question)
print("Intent:", intent)
print("Answer:", answer)

Question: Which player scored for Paraguay against Chile?
Intent: scorer
Answer: The scorers were Braian Samudio, Miguel Almirón in the Chile vs Paraguay match on 2021-06-24.


In [56]:
# Cell 47 - Interactive goalscorer chatbot

def ask_goalscorer_question(question):
    intent = predict_intent(question)

    print(f"\nPredicted intent: {intent}")

    if intent not in GOALSCORER_INTENTS:
        print(
            "\nAssistant: This question does not belong to "
            "the goalscorer section."
        )
        return

    # Initial goal-event candidates
    goal_candidates = find_goalscorer_candidates(question)

    if goal_candidates.empty:
        print(
            "\nAssistant: I couldn't find a goal record "
            "matching your question."
        )
        return

    # Convert goal events into unique possible matches
    unique_matches = get_unique_goal_matches(goal_candidates)

    enriched_matches = enrich_goal_matches(unique_matches)

    print("Possible matches found:", len(enriched_matches))

    # Resolve MATCH ambiguity
    while len(enriched_matches) > 1:

        clarification = generate_goal_match_clarification(
            enriched_matches
        )

        print(f"\nAssistant: {clarification}")

        reply = input("\nYou: ").strip()

        if not reply:
            print(
                "\nAssistant: Please provide another detail "
                "about the match."
            )
            continue

        new_matches = apply_goal_match_clarification(
            enriched_matches,
            reply
        )

        if len(new_matches) == len(enriched_matches):
            print(
                "\nAssistant: That detail didn't narrow the "
                "possible matches. Please try another detail."
            )
            continue

        if len(new_matches) == 0:
            print(
                "\nAssistant: I couldn't find a remaining match "
                "with that detail. Please try something else."
            )
            continue

        enriched_matches = new_matches

        print(
            f"\nRemaining possible matches: "
            f"{len(enriched_matches)}"
        )

    # Exactly one match identified
    selected_match = enriched_matches.iloc[0]
    selected_key = selected_match["match_key"]

    # Return ALL relevant goal events from that match
    selected_goal_events = goal_candidates[
        goal_candidates["match_key"] == selected_key
    ].copy()

    answer = generate_goalscorer_answer(
        intent,
        selected_goal_events
    )

    print(f"\nAssistant: {answer}")

In [57]:
# Cell 48 - Full goalscorer conversation test

question = "Which player scored for Paraguay against Chile?"

ask_goalscorer_question(question)


Predicted intent: scorer
Possible matches found: 32

Assistant: I found matches across 29 different years. Do you remember roughly which year it was?

You: 2022

Assistant: That detail didn't narrow the possible matches. Please try another detail.

Assistant: I found matches across 29 different years. Do you remember roughly which year it was?

You: 2021

Remaining possible matches: 1

Assistant: The scorers were Braian Samudio, Miguel Almirón in the Chile vs Paraguay match on 2021-06-24.


In [58]:
# Cell 49 - Test goal_minute intent

question = "At what minute did Christian Eriksen find the net against Slovenia?"

print("Intent:", predict_intent(question))

goal_candidates = find_goalscorer_candidates(question)

print("Goal records found:", len(goal_candidates))

display(
    goal_candidates[
        [
            "date",
            "home_team",
            "away_team",
            "team",
            "scorer",
            "minute",
            "own_goal",
            "penalty"
        ]
    ].head(20)
)

Intent: goal_minute
Goal records found: 1


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
44962,2024-06-16,Slovenia,Denmark,Denmark,Christian Eriksen,17,False,False


In [59]:
# Cell 50 - Test penalty_status intent

question = "Did Georgios Samaras convert a penalty against Germany on 2012-06-22?"

print("Intent:", predict_intent(question))

goal_candidates = find_goalscorer_candidates(question)

print("Goal records found:", len(goal_candidates))

display(
    goal_candidates[
        [
            "date",
            "home_team",
            "away_team",
            "team",
            "scorer",
            "minute",
            "own_goal",
            "penalty"
        ]
    ]
)

Intent: penalty_status
Goal records found: 1


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
32014,2012-06-22,Germany,Greece,Greece,Georgios Samaras,55,False,False


In [60]:
# Cell 51 - Test own_goal_status intent

question = "Did Jordi Alba score an own goal against Italy?"

print("Intent:", predict_intent(question))

goal_candidates = find_goalscorer_candidates(question)

print("Goal records found:", len(goal_candidates))

display(
    goal_candidates[
        [
            "date",
            "home_team",
            "away_team",
            "team",
            "scorer",
            "minute",
            "own_goal",
            "penalty"
        ]
    ].head(20)
)

Intent: own_goal_status
Goal records found: 1


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
32025,2012-07-01,Spain,Italy,Spain,Jordi Alba,41,False,False


In [61]:
# Cell 52 - Improved interactive goalscorer chatbot

def ask_goalscorer_question(question):
    intent = predict_intent(question)

    print(f"\nPredicted intent: {intent}")

    if intent not in GOALSCORER_INTENTS:
        print(
            "\nAssistant: This question does not belong to "
            "the goalscorer section."
        )
        return

    # Initial goal-event candidates
    goal_candidates = find_goalscorer_candidates(question)

    if goal_candidates.empty:
        print(
            "\nAssistant: I couldn't find a goal record "
            "matching your question."
        )
        return

    # Convert goal events into unique matches
    unique_matches = get_unique_goal_matches(goal_candidates)
    enriched_matches = enrich_goal_matches(unique_matches)

    print("Possible matches found:", len(enriched_matches))

    # Resolve match ambiguity
    while len(enriched_matches) > 1:
        clarification = generate_goal_match_clarification(
            enriched_matches
        )

        print(f"\nAssistant: {clarification}")

        reply = input("\nYou: ").strip()

        if not reply:
            print(
                "\nAssistant: Please provide another detail "
                "about the match."
            )
            continue

        new_matches = apply_goal_match_clarification(
            enriched_matches,
            reply
        )

        if len(new_matches) == len(enriched_matches):
            print(
                "\nAssistant: That detail didn't narrow the "
                "possible matches. Please try another detail."
            )
            continue

        if len(new_matches) == 0:
            print(
                "\nAssistant: I couldn't find a remaining match "
                "with that detail. Please try something else."
            )
            continue

        enriched_matches = new_matches

        print(
            f"\nRemaining possible matches: "
            f"{len(enriched_matches)}"
        )

    # Exactly one match remains
    selected_match = enriched_matches.iloc[0]
    selected_key = selected_match["match_key"]

    selected_goal_events = goal_candidates[
        goal_candidates["match_key"] == selected_key
    ].copy()

    answer = generate_goalscorer_answer(
        intent,
        selected_goal_events
    )

    print(f"\nAssistant: {answer}")

In [62]:
ask_goalscorer_question(
    "At what minute did Christian Eriksen find the net against Slovenia?"
)


Predicted intent: goal_minute
Possible matches found: 1

Assistant: The matching goal(s) were scored by Christian Eriksen in the 17th minute.


In [63]:
ask_goalscorer_question(
    "Did Georgios Samaras convert a penalty against Germany on 2012-06-22?"
)


Predicted intent: penalty_status
Possible matches found: 1

Assistant: Georgios Samaras's goal was not a penalty.


In [64]:
ask_goalscorer_question(
    "Did Jordi Alba score an own goal against Italy?"
)


Predicted intent: own_goal_status
Possible matches found: 1

Assistant: Jordi Alba's goal was not an own goal.


In [65]:
# Cell 53 - Prepare shootout data

SHOOTOUT_INTENTS = {
    "shootout_winner",
    "first_shooter"
}

shootouts_with_key = add_match_key(shootouts_df)

print("Shootout records:", len(shootouts_with_key))
print("\nColumns:")
print(shootouts_with_key.columns.tolist())

Shootout records: 682

Columns:
['date', 'home_team', 'away_team', 'winner', 'first_shooter', 'match_key']


In [66]:
# Cell 54 - Extract entities for shootout questions

def extract_shootout_entities(question):
    question_normalized = normalize_text(question)

    found_teams = [
        team
        for team in all_teams
        if normalize_text(team) in question_normalized
    ]

    year_match = re.search(
        r"\b(18|19|20)\d{2}\b",
        question
    )

    year = year_match.group() if year_match else None

    return {
        "teams": found_teams,
        "year": year
    }

In [67]:
# Cell 55 - Find shootout candidates

def find_shootout_candidates(question):
    entities = extract_shootout_entities(question)

    candidates = shootouts_with_key.copy()

    teams = entities["teams"]
    year = entities["year"]

    # Two teams mentioned
    if len(teams) >= 2:
        team1, team2 = teams[:2]

        candidates = candidates[
            (
                (candidates["home_team"] == team1)
                & (candidates["away_team"] == team2)
            )
            |
            (
                (candidates["home_team"] == team2)
                & (candidates["away_team"] == team1)
            )
        ]

    # Year
    if year:
        candidates = candidates[
            candidates["date"]
            .astype(str)
            .str.startswith(year)
        ]

    return candidates

In [68]:
# Cell 56 - Test shootout retrieval

question = (
    "Who took the first penalty in the shootout "
    "between Uruguay and Ghana?"
)

intent = predict_intent(question)

shootout_candidates = find_shootout_candidates(
    question
)

print("Predicted intent:", intent)
print(
    "Possible shootout records:",
    len(shootout_candidates)
)

display(
    shootout_candidates[
        [
            "date",
            "home_team",
            "away_team",
            "winner",
            "first_shooter"
        ]
    ]
)

Predicted intent: first_shooter
Possible shootout records: 1


,date,home_team,away_team,winner,first_shooter
409,2010-07-02,Uruguay,Ghana,Uruguay,Uruguay


In [69]:
# Cell 57 - Enrich shootout records with results data

def enrich_shootout_matches(shootout_candidates):
    return shootout_candidates.merge(
        results_with_key[
            [
                "match_key",
                "home_score",
                "away_score",
                "tournament",
                "city",
                "country",
                "neutral"
            ]
        ],
        on="match_key",
        how="left"
    )

In [70]:
enriched_shootouts = enrich_shootout_matches(
    shootout_candidates
)

display(
    enriched_shootouts[
        [
            "date",
            "home_team",
            "away_team",
            "winner",
            "first_shooter",
            "tournament",
            "city",
            "country"
        ]
    ]
)

,date,home_team,away_team,winner,first_shooter,tournament,city,country
0,2010-07-02,Uruguay,Ghana,Uruguay,Uruguay,FIFA World Cup,Johannesburg,South Africa


In [71]:
# Cell 58 - Generate answers for shootout intents

def generate_shootout_answer(intent, row):
    home_team = row["home_team"]
    away_team = row["away_team"]
    date = row["date"]

    if intent == "shootout_winner":
        winner = row["winner"]

        return (
            f"{winner} won the penalty shootout "
            f"between {home_team} and {away_team} on {date}."
        )

    elif intent == "first_shooter":
        first_shooter = row["first_shooter"]

        if pd.isna(first_shooter) or str(first_shooter).strip().lower() == "unknown":
            return (
                f"The first shooter is not recorded for the "
                f"{home_team} vs {away_team} shootout on {date}."
            )

        return (
            f"{first_shooter} took the first penalty in the "
            f"{home_team} vs {away_team} shootout on {date}."
        )

    return "Answer generation for this shootout intent is not available yet."

In [72]:
question = (
    "Who took the first penalty in the shootout "
    "between Uruguay and Ghana?"
)

intent = predict_intent(question)

shootout_candidates = find_shootout_candidates(question)

enriched_shootouts = enrich_shootout_matches(
    shootout_candidates
)

answer = generate_shootout_answer(
    intent,
    enriched_shootouts.iloc[0]
)

print("Question:", question)
print("Intent:", intent)
print("Answer:", answer)

Question: Who took the first penalty in the shootout between Uruguay and Ghana?
Intent: first_shooter
Answer: Uruguay took the first penalty in the Uruguay vs Ghana shootout on 2010-07-02.


In [73]:
question = (
    "Who won the penalty shootout between Uruguay and Ghana?"
)

intent = predict_intent(question)

shootout_candidates = find_shootout_candidates(question)

enriched_shootouts = enrich_shootout_matches(
    shootout_candidates
)

answer = generate_shootout_answer(
    intent,
    enriched_shootouts.iloc[0]
)

print("Question:", question)
print("Intent:", intent)
print("Answer:", answer)

Question: Who won the penalty shootout between Uruguay and Ghana?
Intent: shootout_winner
Answer: Uruguay won the penalty shootout between Uruguay and Ghana on 2010-07-02.


In [74]:
# Cell 60 - Find team pairs with multiple penalty shootouts

shootout_pairs = shootouts_with_key.copy()

shootout_pairs["team_pair"] = shootout_pairs.apply(
    lambda row: tuple(
        sorted([row["home_team"], row["away_team"]])
    ),
    axis=1
)

repeated_shootout_pairs = (
    shootout_pairs
    .groupby("team_pair")
    .size()
    .sort_values(ascending=False)
)

repeated_shootout_pairs = repeated_shootout_pairs[
    repeated_shootout_pairs > 1
]

print(
    "Team pairs with multiple shootouts:",
    len(repeated_shootout_pairs)
)

print("\nTop repeated shootout matchups:\n")
print(repeated_shootout_pairs.head(20))

Team pairs with multiple shootouts: 100

Top repeated shootout matchups:

team_pair
(Indonesia, Thailand)       5
(Guinea, Mali)              5
(Guernsey, Jersey)          5
(Kenya, Uganda)             5
(Brazil, Uruguay)           4
(Botswana, South Africa)    4
(Costa Rica, Honduras)      4
(Japan, South Korea)        4
(Zambia, Zimbabwe)          4
(Argentina, Brazil)         4
(Malawi, South Africa)      4
(Egypt, Ivory Coast)        3
(Lesotho, Mozambique)       3
(Malawi, Zambia)            3
(Argentina, Netherlands)    3
(Argentina, Colombia)       3
(Malaysia, Thailand)        3
(Panama, United States)     3
(Costa Rica, Mexico)        3
(Nigeria, Tunisia)          3
dtype: int64


In [75]:
# Cell 61 - Inspect the most repeated shootout matchup

test_pair = repeated_shootout_pairs.index[0]

team1, team2 = test_pair

print("Testing:", team1, "vs", team2)

test_shootouts = shootouts_with_key[
    (
        (shootouts_with_key["home_team"] == team1)
        & (shootouts_with_key["away_team"] == team2)
    )
    |
    (
        (shootouts_with_key["home_team"] == team2)
        & (shootouts_with_key["away_team"] == team1)
    )
]

enriched_test_shootouts = enrich_shootout_matches(
    test_shootouts
)

display(
    enriched_test_shootouts[
        [
            "date",
            "home_team",
            "away_team",
            "winner",
            "first_shooter",
            "tournament",
            "city",
            "country"
        ]
    ]
)

Testing: Indonesia vs Thailand


,date,home_team,away_team,winner,first_shooter,tournament,city,country
0,1979-09-29,Indonesia,Thailand,Indonesia,Unknown,Southeast Asian Games,Jakarta,Indonesia
1,1989-08-30,Indonesia,Thailand,Indonesia,Unknown,Southeast Asian Games,Kuala Lumpur,Malaysia
2,1991-12-04,Indonesia,Thailand,Indonesia,Unknown,Southeast Asian Games,Manila,Philippines
3,1997-10-18,Indonesia,Thailand,Thailand,Unknown,Southeast Asian Games,Jakarta,Indonesia
4,1998-09-05,Indonesia,Thailand,Indonesia,Unknown,AFF Championship,Ho Chi Minh City,Vietnam


In [76]:
# Cell 62 - Smart clarification for shootout candidates

def generate_shootout_clarification(enriched_shootouts):
    if len(enriched_shootouts) == 0:
        return "I couldn't find a matching shootout."

    if len(enriched_shootouts) == 1:
        return None

    return generate_smart_clarification(enriched_shootouts)

In [77]:
# Cell 63 - Apply clarification to shootout candidates

def apply_shootout_clarification(enriched_shootouts, reply):
    return apply_smart_clarification(
        enriched_shootouts,
        reply
    )

In [78]:
# Cell 64 - Interactive shootout chatbot

def ask_shootout_question(question):
    intent = predict_intent(question)

    print(f"\nPredicted intent: {intent}")

    if intent not in SHOOTOUT_INTENTS:
        print(
            "\nAssistant: This question does not belong to "
            "the shootout section."
        )
        return

    shootout_candidates = find_shootout_candidates(question)

    if shootout_candidates.empty:
        print(
            "\nAssistant: I couldn't find a penalty shootout "
            "matching your question."
        )
        return

    enriched_shootouts = enrich_shootout_matches(
        shootout_candidates
    )

    print(
        "Possible shootout records:",
        len(enriched_shootouts)
    )

    while len(enriched_shootouts) > 1:
        clarification = generate_shootout_clarification(
            enriched_shootouts
        )

        print(f"\nAssistant: {clarification}")

        reply = input("\nYou: ").strip()

        if not reply:
            print(
                "\nAssistant: Please provide another detail "
                "about the shootout."
            )
            continue

        new_shootouts = apply_shootout_clarification(
            enriched_shootouts,
            reply
        )

        if len(new_shootouts) == len(enriched_shootouts):
            print(
                "\nAssistant: That detail didn't narrow the "
                "possible shootouts. Please try another detail."
            )
            continue

        if len(new_shootouts) == 0:
            print(
                "\nAssistant: I couldn't match that detail to "
                "the remaining shootouts."
            )
            continue

        enriched_shootouts = new_shootouts

        print(
            f"\nRemaining shootouts: "
            f"{len(enriched_shootouts)}"
        )

    selected_shootout = enriched_shootouts.iloc[0]

    answer = generate_shootout_answer(
        intent,
        selected_shootout
    )

    print(f"\nAssistant: {answer}")

In [79]:
  # Cell 65 - Full shootout ambiguity test

question = (
    "Who won the penalty shootout between "
    "Indonesia and Thailand?"
)

ask_shootout_question(question)


Predicted intent: shootout_winner
Possible shootout records: 5

Assistant: I found matches from multiple years. Do you remember the year? Options: 1979, 1989, 1991, 1997, 1998

You: 1998

Remaining shootouts: 1

Assistant: Indonesia won the penalty shootout between Indonesia and Thailand on 1998-09-05.
